# Dataset Creation
Builds the FFT magnitude dataset from the LibriSpeech audio files, applies the cipher permutation, splits into train/test, and saves everything to `.npy` files for use by `train_model.ipynb`.

In [1]:
import os
import numpy as np
import librosa
import glob
import scipy.fft as fft

## 1. Load audio files

In [2]:
audio_paths = glob.glob('dev-clean/LibriSpeech/dev-clean/**/*.flac', recursive=True)
print(f"Found {len(audio_paths)} .flac files.")

target_sr = 17000
chunk_size = 25000
base_frequencies = fft.rfftfreq(chunk_size, 1/target_sr)[:-1]
rfft_size = len(base_frequencies) 

Found 2703 .flac files.


## 2. Chunk each file and compute FFT magnitudes

In [3]:
all_magnitudes = []
all_frequencies = []

for video_path in audio_paths: 
    y, sr = librosa.load(video_path, sr=target_sr)
    N = len(y)
    
    audio_parts = np.array([y[x:x+chunk_size] for x in range(0, N, chunk_size) if x+chunk_size <= N])
    if len(audio_parts) == 0:
        continue
        
    fourier_transforms = np.array([fft.rfft(part) for part in audio_parts])
    
    fourier_magnitudes = np.abs(fourier_transforms)[:, :-1]
    
    fourier_magnitudes = fourier_magnitudes.astype(np.float32)
    
    all_magnitudes.append(fourier_magnitudes)
    all_frequencies.append(np.tile(base_frequencies, len(audio_parts)).astype(np.float32))

total_magnitudes = np.concatenate(all_magnitudes)
total_frequencies = np.concatenate(all_frequencies).reshape(-1, rfft_size)

print(f"Total magnitudes shape: {total_magnitudes.shape}")

c:\Users\yashd\Stuff\Encryption_Decryption-SOC-26\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total magnitudes shape: (11813, 12500)


## 3. Cipher: permute magnitude sub-blocks

In [4]:
class cipher:
    def __init__(self, n):
        self.n = n
    
    def division(self, magnitudes):
        length = len(magnitudes)
        sub_magnitudes = [magnitudes[x*(length//self.n):(x+1)*(length//self.n)] for x in range(self.n)]
        permut_rand = np.random.permutation(self.n)
        permut_magnitudes = [sub_magnitudes[x] for x in permut_rand]
        net_permut_mag = np.concatenate(permut_magnitudes)
        return net_permut_mag, permut_rand
    
my_cipher = cipher(20)
total_permuted_mag = []
total_rand = []

for each in total_magnitudes:
    net_permut_mag, permut_rand = my_cipher.division(each)
    total_permuted_mag.append(net_permut_mag)
    total_rand.append(permut_rand)

total_permuted_mag = np.array(total_permuted_mag, dtype=np.float32)

## 4. Train / test split

In [5]:
split_idx = int(0.8 * len(total_magnitudes))

train_original = total_magnitudes[:split_idx]
train_ciphered = total_permuted_mag[:split_idx]
train_frequencies = total_frequencies[:split_idx]

test_original = total_magnitudes[split_idx:]
test_ciphered = total_permuted_mag[split_idx:]
test_frequencies = total_frequencies[split_idx:]

## 5. Save dataset arrays to `.npy` files

In [6]:
save_dir = "dataset_files"
os.makedirs(save_dir, exist_ok=True)

np.save(os.path.join(save_dir, "train_original.npy"), train_original)
np.save(os.path.join(save_dir, "train_ciphered.npy"), train_ciphered)
np.save(os.path.join(save_dir, "train_frequencies.npy"), train_frequencies)
np.save(os.path.join(save_dir, "test_original.npy"), test_original)
np.save(os.path.join(save_dir, "test_ciphered.npy"), test_ciphered)
np.save(os.path.join(save_dir, "test_frequencies.npy"), test_frequencies)
np.save(os.path.join(save_dir, "base_frequencies.npy"), base_frequencies)
np.save(os.path.join(save_dir, "audio_paths.npy"), np.array(audio_paths, dtype=object), allow_pickle=True)

print(f"Saved dataset .npy files to '{save_dir}/'")

Saved dataset .npy files to 'dataset_files/'
